In [1]:
import os 
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async" 
import tensorflow as tf 
tf.keras.backend.clear_session() 
import pandas as pd 
from deepmreye import analyse, architecture, preprocess, train 
from deepmreye.util import data_generator, model_opts, util 
import numpy as np 

gpus = tf.config.experimental.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(gpus[0], True)

2026-02-05 07:58:55.611146: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 07:58:55.666148: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-05 07:58:55.666190: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-05 07:58:55.669186: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-05 07:58:55.681780: I tensorflow/core/platform/cpu_feature_guar

In [2]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    # If train_list and test_list are provided, use them directly
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        # Otherwise, build from datasets
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    # Call create_generators with the final lists
    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

In [3]:
opts = model_opts.get_opts()
print(opts)

{'kernel': 3, 'lr': 2e-05, 'filters': 32, 'multiplier': 2, 'depth': 4, 'dropout_rate': 0.1, 'num_dense': 2, 'num_fc': 1024, 'gaussian_noise': 0, 'activation': <function mish at 0x7fe811c3fe20>, 'groups': 8, 'inner_timesteps': 10, 'loss_euclidean': 1, 'loss_confidence': 0.1, 'epochs': 25, 'steps_per_epoch': 1500, 'validation_steps': 1500, 'train_test_split': 0.6, 'batch_size': 8, 'mixed_batches': True, 'mc_dropout': False, 'rotation_x': 5, 'rotation_y': 5, 'rotation_z': 5, 'shift': 4, 'zoom': 0.15}


In [15]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',dtype=str)

In [4]:
import glob

npz_files = glob.glob("/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/*.npz")

filtered_files = [
    f for f in npz_files
    if "right" not in f.lower() and "left" not in f.lower()
]

for f in filtered_files:
    print(f)


/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9087.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9091.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9001.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9007.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9008.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9013.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9023.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9025.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9029.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9033.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9041.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9048.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9050.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9053.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9055.npz
/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9

In [5]:
# Load the trained model 
generators = data_generator.create_generators(filtered_files,
                                              filtered_files)
generators = (*generators, filtered_files, filtered_files
              )  
(model_full, model_inference) = train.train_model(dataset="prediction",
                                             generators=generators,
                                             opts=opts,
                                             return_untrained=True)
model_inference.load_weights('/mnt/compneuro/Neuronus/datasets_1to5.h5')

Training set (/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti) contains 29 subjects: 
['9087', '9091', '9001', '9007', '9008', '9013', '9023', '9025', '9029', '9033',
 '9041', '9048', '9050', '9053', '9055', '9057', '9063', '9070', '9071', '9074',
 '9077', '9082', '9084', '9100', '9081', '9083', '9085', '9088', '9092']
Test set (/mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti) contains 29 subjects: 
['9087', '9091', '9001', '9007', '9008', '9013', '9023', '9025', '9029', '9033',
 '9041', '9048', '9050', '9053', '9055', '9057', '9063', '9070', '9071', '9074',
 '9077', '9082', '9084', '9100', '9081', '9083', '9085', '9088', '9092']


2026-02-05 07:59:05.014912: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-05 07:59:05.016564: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-05 07:59:05.017854: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [6]:
(evaluation_ft, scores_ft) = train.evaluate_model(
    dataset='fine_tuned_resting_state', 
    model=model_inference, 
    generators=generators, 
    save=False, 
    model_description='fine_tuned_on_rs', 
    verbose=2
)

2026-02-05 07:59:11.212147: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


1 / 29 - Model Performance for /mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9087.npz
              Pearson             R^2-Score               Eucl. Error             
                    X     Y  Mean         X      Y   Mean        Mean Median   Std
Default        -0.328 0.425 0.049    -2.895 -0.666 -1.780       3.211  3.250 1.654
Default subTR  -0.265 0.378 0.057    -2.128 -0.470 -1.299       3.352  3.308 1.722
Refined        -0.328 0.425 0.049    -2.895 -0.666 -1.780       3.211  3.250 1.654
Refined subTR  -0.265 0.378 0.057    -2.128 -0.470 -1.299       3.352  3.308 1.722


2 / 29 - Model Performance for /mnt/compneuro/deepmreye_finetuning/sleepybrain/nifti/9091.npz
              Pearson               R^2-Score               Eucl. Error             
                    X      Y   Mean         X      Y   Mean        Mean Median   Std
Default         0.024 -0.286 -0.131    -0.353 -1.372 -0.862       3.651  3.476 1.700
Default subTR   0.015 -0.251 -0.118    -0.321 -1.073 -0.6

In [7]:
fig = analyse.visualise_predictions_slider(
    evaluation_ft, 
    scores_ft, 
    color="rgb(0, 150, 175)", 
    bg_color="rgb(255,255,255)",
    ylim=[-10, 10],
)
fig.show()

FigureWidget({
    'data': [{'boxpoints': 'all',
              'fillcolor': 'rgb(180, 180, 180)',
              'line': {'color': 'rgb(0,0,0)'},
              'marker': {'color': 'rgb(0, 150, 175)',
                         'line': {'color': 'rgb(0,0,0)', 'width': 2},
                         'opacity': 0.65,
                         'size': 12},
              'name': 'Default',
              'pointpos': 0,
              'text': [participant 9087, participant 9091, participant 9001,
                       participant 9007, participant 9008, participant 9013,
                       participant 9023, participant 9025, participant 9029,
                       participant 9033, participant 9041, participant 9048,
                       participant 9050, participant 9053, participant 9055,
                       participant 9057, participant 9063, participant 9070,
                       participant 9071, participant 9074, participant 9077,
                       participant 9082, participa

# LETS GO 

In [8]:
example_data=np.load('/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9001.npz')

In [9]:
print(example_data['data_0'].shape) # this is the 3D eye data of a single timepoint
print(example_data['label_0'].shape) # this is the 10 gaze positions for that TR (x,y). you need to get the median of these to have a single gaze pozition for TR
print(example_data['identifier_0']) # this is the subject ID 

# the last data is example_data['data_181'], showing the 182 TRs 

(47, 29, 18)
(10, 2)
['9001' '6']


In [10]:
import os
import numpy as np

data_dir = '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/'
files = os.listdir(data_dir)

all_data = []
all_labels = []
all_subjects = []

for file in files:
    npz = np.load(os.path.join(data_dir, file))
    
    # Loop over all TRs in the file
    tr_keys = [k for k in npz.keys() if k.startswith('data_')]
    for key in tr_keys:
        data = npz[key]  # shape (47, 29, 18)
        label = npz[key.replace('data', 'label')]  # shape (10, 2)
        median_label = np.median(label, axis=0)    # shape (2,)
        identifier = npz[key.replace('data', 'identifier')][0]  # e.g., '9001'
        
        all_data.append(data)
        all_labels.append(median_label)
        all_subjects.append(identifier)

# Convert to arrays
X = np.stack(all_data, axis=0)       # shape (num_TRs, 47, 29, 18)
y = np.stack(all_labels, axis=0)     # shape (num_TRs, 2)
subject_ids = np.array(all_subjects) # shape (num_TRs,)

# Add channel dimension for Conv3D
Xdd = X[..., np.newaxis]               # shape (num_TRs, 47, 29, 18, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("subject_ids shape:", subject_ids.shape)


X shape: (5278, 47, 29, 18)
y shape: (5278, 2)
subject_ids shape: (5278,)


In [11]:
import numpy as np

def create_sequences(X, y, subject_ids, seq_len=10):
    """
    Convert single-TR data into sequences of `seq_len` TRs for the model.
    Args:
        X: np.array, shape (num_TRs, 47, 29, 18, 1)
        y: np.array, shape (num_TRs, 2)  # median gaze per TR
        subject_ids: np.array, shape (num_TRs,)
        seq_len: int, number of TRs per sample (inner_timesteps)

    Returns:
        X_seq: np.array, shape (num_sequences, seq_len, 47,29,18,1)
        y_seq: np.array, shape (num_sequences, seq_len, 2)
        subject_seq: np.array, shape (num_sequences,)
    """
    X_seq, y_seq, subject_seq = [], [], []

    # group by subject to avoid crossing sequences between subjects
    unique_subjects = np.unique(subject_ids)
    for subj in unique_subjects:
        subj_mask = subject_ids == subj
        X_subj = X[subj_mask]
        y_subj = y[subj_mask]

        n_tr = len(X_subj)
        # sliding window
        for start in range(n_tr - seq_len + 1):
            X_seq.append(X_subj[start:start+seq_len])
            y_seq.append(y_subj[start:start+seq_len])
            subject_seq.append(subj)

    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    subject_seq = np.array(subject_seq)
    return X_seq, y_seq, subject_seq

# Example usage
seq_len = 10  # as expected by the model
X_seq, y_seq, subject_seq = create_sequences(X, y, subject_ids, seq_len=seq_len)

print("X_seq shape:", X_seq.shape)       # (num_samples, 10, 47,29,18,1)
print("y_seq shape:", y_seq.shape)       # (num_samples, 10, 2)
print("subject_seq shape:", subject_seq.shape)


X_seq shape: (5017, 10, 47, 29, 18)
y_seq shape: (5017, 10, 2)
subject_seq shape: (5017,)


In [17]:
# =========================
# Step 1: Rebuild the original model architecture
# (same as the one used for pretraining)
# =========================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, Dropout, AveragePooling3D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

input_layer = Input(shape=(47, 29, 18, 1))

x = Conv3D(32, (3,3,3), padding='same', activation='relu')(input_layer)
x = Dropout(0.2)(x)
x = Conv3D(32, (3,3,3), padding='same', activation='relu')(x)
x = AveragePooling3D()(x)
x = Flatten()(x)
x = Dense(1024, activation='relu')(x)

# New output layer for median gaze
output = Dense(2, activation='linear', name='gaze_xy')(x)

model_inference = Model(inputs=input_layer, outputs=output)

# =========================
# Step 2: Load pretrained weights
# =========================

model_inference.load_weights('/mnt/compneuro/Neuronus/datasets_1to5.h5', by_name=True)

# =========================
# Step 3: Freeze layers except last
# =========================
for layer in model_inference.layers[:-2]:
    layer.trainable = False

# =========================
# Step 4: Compile
# =========================
model_inference.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])


In [92]:
import tensorflow as tf
from tensorflow.keras import backend as K

# Clear any existing models
K.clear_session()

# Optional: force garbage collection
import gc
gc.collect()


47847

In [7]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.reset_memory_stats(gpu)
    except Exception as e:
        print(e)


TFE_ResetMemoryStats(): incompatible function arguments. The following argument types are supported:
    1. (arg0: handle, arg1: str) -> None

Invoked with: <capsule object NULL at 0x7f718c9d3450>, PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [13]:
model_inference

In [15]:
model_inference.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss=gaze_composite_loss,
    metrics=['mae']
)


In [64]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from deepmreye import train, data_generator
from sklearn.metrics import r2_score

# =========================
# Subject-level split


ImportError: cannot import name 'data_generator' from 'deepmreye' (/home/CemalKoba/venvs/deepmreye-gpu/lib/python3.10/site-packages/deepmreye/__init__.py)

In [44]:
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 10, 47, 29, 18,   0         
                              1)]                                
                                                                 
 time_distributed (TimeDist  (None, 10, 47, 29, 18,    896       
 ributed)                    32)                                 
                                                                 
 time_distributed_1 (TimeDi  (None, 10, 47, 29, 18,    0         
 stributed)                  32)                                 
                                                                 
 time_distributed_2 (TimeDi  (None, 10, 47, 29, 18,    27680     
 stributed)                  32)                                 
                                                                 
 time_distributed_3 (TimeDi  (None, 10, 23, 14, 9, 3   0   

In [42]:
subj_train

array(['9001', '9001', '9001', ..., '9092', '9092', '9092'], dtype='<U4')

Leave one out 

In [96]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv3D, Dropout, AveragePooling3D,
                                     Flatten, Dense)
from tensorflow.keras.optimizers import Adam

# =========================
# Smooth L1 (Huber) loss
# =========================
def smooth_l1_loss(y_true, y_pred, delta=0.03):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic
    loss = 0.5 * tf.square(quadratic) + delta * linear
    return tf.reduce_mean(loss)

# =========================
# Model builder
# =========================
def build_model(input_shape=(47,29,18,1), dense_units=256):
    inp = Input(shape=input_shape)
    x = Conv3D(32, (3,3,3), padding='same', activation='relu')(inp)
    x = Dropout(0.1)(x)
    x = Conv3D(32, (3,3,3), padding='same', activation='relu')(x)
    x = AveragePooling3D()(x)
    x = Flatten()(x)
    x = Dense(dense_units, activation='relu')(x)
    out = Dense(2, activation='linear', name='gaze_xy')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=Adam(9.85e-6), loss=smooth_l1_loss, metrics=['mae'])
    return model

# =========================
# Leave-One-Subject-Out (LOSO) training
# =========================
unique_subjects = np.unique(subject_ids)
results = []

for test_subj in unique_subjects:
    print(f"\n=== Testing on Subject {test_subj} ===")
    
    # Train/test split by subject
    train_idx = subject_ids != test_subj
    test_idx  = subject_ids == test_subj
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test   = X[test_idx], y[test_idx]
    
    # Build and train a fresh model
    model = build_model(input_shape=X_train.shape[1:])
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=8,
        verbose=0
    )
    
    # Predict
    y_pred = model.predict(X_test, batch_size=16)
    
    # Metrics
    corr_x = np.corrcoef(y_test[:,0], y_pred[:,0])[0,1]
    corr_y = np.corrcoef(y_test[:,1], y_pred[:,1])[0,1]
    r2_x = r2_score(y_test[:,0], y_pred[:,0])
    r2_y = r2_score(y_test[:,1], y_pred[:,1])
    
    print(f"Pearson X: {corr_x:.3f}, Y: {corr_y:.3f} | R2 X: {r2_x:.3f}, Y: {r2_y:.3f}")
    
    results.append([test_subj, corr_x, corr_y, r2_x, r2_y])
    
    # Plot
    fig, axes = plt.subplots(1,2, figsize=(14,4))
    axes[0].plot(y_test[:,0], label='True X', color='steelblue', alpha=0.8)
    axes[0].plot(y_pred[:,0], label='Pred X', color='lightblue', alpha=0.8)
    axes[0].set_title(f'Subject {test_subj} - X')
    axes[0].legend()
    
    axes[1].plot(y_test[:,1], label='True Y', color='indianred', alpha=0.8)
    axes[1].plot(y_pred[:,1], label='Pred Y', color='salmon', alpha=0.8)
    axes[1].set_title(f'Subject {test_subj} - Y')
    axes[1].legend()
    
    plt.show()

# =========================
# Summary
# =========================
import pandas as pd
results_df = pd.DataFrame(results, columns=['subject_id','pearson_x','pearson_y','r2_x','r2_y'])
print("\n=== LOSO Summary ===")
print(results_df)



=== Testing on Subject 9001 ===


ValueError: Input 0 of layer "average_pooling3d_6" is incompatible with the layer: expected ndim=5, found ndim=8. Full shape received: (None, 1, 1, 47, 29, 18, 1, 32)

In [94]:
from tensorflow.keras.layers import TimeDistributed, LSTM

def build_model_with_lstm(input_shape=(47,29,18,1),
                          dense_units=128,
                          lstm_units=64,
                          dropout_rate=0.3,
                          l2_reg=1e-4,
                          pretrained_path=None):
    """
    Builds a 3D-CNN + LSTM model for gaze prediction with optional pretrained weights.
    input_shape: (time, height, width, depth, channels)
    """
    inp = Input(shape=input_shape)
    
    # TimeDistributed 3D conv layers
    x = TimeDistributed(Conv3D(16, (3,3,3), padding='same', activation='relu',
                               kernel_regularizer=l2(l2_reg)), name='conv1_td')(inp)
    x = TimeDistributed(Dropout(dropout_rate), name='dropout1_td')(x)
    x = TimeDistributed(Conv3D(16, (3,3,3), padding='same', activation='relu',
                               kernel_regularizer=l2(l2_reg)), name='conv2_td')(x)
    x = TimeDistributed(AveragePooling3D(), name='avgpool_td')(x)
    x = TimeDistributed(Flatten(), name='flatten_td')(x)
    
    # LSTM over timesteps
    x = LSTM(lstm_units, activation='tanh', name='lstm')(x)
    
    # Dense head
    x = Dense(dense_units, activation='relu', kernel_regularizer=l2(l2_reg), name='dense1')(x)
    x = Dropout(dropout_rate, name='dropout2')(x)
    out = Dense(2, activation='linear', name='gaze_xy')(x)
    
    model = Model(inputs=inp, outputs=out)
    
    # Load pretrained weights if available (conv layers only)
    if pretrained_path is not None:
        print(f"Loading pretrained weights from {pretrained_path}")
        temp_model = build_model(input_shape=input_shape[1:],  # exclude temporal dim
                                 dense_units=dense_units,
                                 dropout_rate=dropout_rate,
                                 l2_reg=l2_reg)
        temp_model.load_weights(pretrained_path)
        # Copy conv weights to TimeDistributed layers
        model.get_layer('conv1_td').layer.set_weights(temp_model.get_layer('conv1').get_weights())
        model.get_layer('conv2_td').layer.set_weights(temp_model.get_layer('conv2').get_weights())
        # Freeze conv layers
        model.get_layer('conv1_td').trainable = False
        model.get_layer('conv2_td').trainable = False
    
    model.compile(optimizer=Adam(9.85e-6), loss=smooth_l1_loss, metrics=['mae'])
    return model


In [95]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, Dropout, AveragePooling3D, Flatten, Dense, TimeDistributed, LSTM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
def build_model(input_shape=(47,29,18,1), dense_units=256):
    inp = Input(shape=input_shape)
    x = Conv3D(32, (3,3,3), padding='same', activation='relu')(inp)
    x = Dropout(0.1)(x)
    x = Conv3D(32, (3,3,3), padding='same', activation='relu')(x)
    x = AveragePooling3D()(x)
    x = Flatten()(x)
    x = Dense(dense_units, activation='relu')(x)
    out = Dense(2, activation='linear', name='gaze_xy')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=Adam(9.85e-6), loss=smooth_l1_loss, metrics=['mae'])
    return model

# =========================
# Smooth L1 (Huber) loss
# =========================
def smooth_l1_loss(y_true, y_pred, delta=0.03):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic
    loss = 0.5 * tf.square(quadratic) + delta * linear
    return tf.reduce_mean(loss)

# =========================
# LSTM + TimeDistributed CNN model
# =========================
def build_model_with_lstm(input_shape=(47,29,18,1),
                          dense_units=128,
                          lstm_units=64,
                          dropout_rate=0.3,
                          l2_reg=1e-4,
                          pretrained_path=None):
    inp = Input(shape=input_shape)
    
    # TimeDistributed 3D conv layers
    x = TimeDistributed(Conv3D(16, (3,3,3), padding='same', activation='relu',
                               kernel_regularizer=l2(l2_reg)), name='conv1_td')(inp)
    x = TimeDistributed(Dropout(dropout_rate), name='dropout1_td')(x)
    x = TimeDistributed(Conv3D(16, (3,3,3), padding='same', activation='relu',
                               kernel_regularizer=l2(l2_reg)), name='conv2_td')(x)
    x = TimeDistributed(AveragePooling3D(), name='avgpool_td')(x)
    x = TimeDistributed(Flatten(), name='flatten_td')(x)
    
    # LSTM over timesteps
    x = LSTM(lstm_units, activation='tanh', name='lstm')(x)
    
    # Dense head
    x = Dense(dense_units, activation='relu', kernel_regularizer=l2(l2_reg), name='dense1')(x)
    x = Dropout(dropout_rate, name='dropout2')(x)
    out = Dense(2, activation='linear', name='gaze_xy')(x)
    
    model = Model(inputs=inp, outputs=out)
    
    # Load pretrained weights if available (conv layers only)
    if pretrained_path is not None:
        print(f"Loading pretrained weights from {pretrained_path}")
        # Build the original CNN model (no temporal dim)
        temp_model = build_model(input_shape=input_shape[1:],  # exclude temporal dim
                                 dense_units=dense_units,
                                 dropout_rate=dropout_rate,
                                 l2_reg=l2_reg)
        temp_model.load_weights(pretrained_path)
        # Copy conv weights
        model.get_layer('conv1_td').layer.set_weights(temp_model.get_layer('conv1').get_weights())
        model.get_layer('conv2_td').layer.set_weights(temp_model.get_layer('conv2').get_weights())
        # Freeze conv layers
        model.get_layer('conv1_td').trainable = False
        model.get_layer('conv2_td').trainable = False
    
    model.compile(optimizer=Adam(9.85e-6), loss=smooth_l1_loss, metrics=['mae'])
    return model

# =========================
# LOSO training loop
# =========================
results = []
unique_subjects = np.unique(subject_ids)
# Add channel and time dimensions
X = X[..., np.newaxis]       # add channels
X = X[:, np.newaxis, ...]    # add temporal dimension (timesteps=1 if needed)

for test_subj in unique_subjects:
    train_idx = subject_ids != test_subj
    test_idx  = subject_ids == test_subj
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test   = X[test_idx], y[test_idx]
    
    model = build_model_with_lstm(input_shape=X_train.shape[1:], pretrained_path='datasets_1to5.h5')

for test_subj in unique_subjects:
    print(f"\n=== Testing on Subject {test_subj} ===")
    
    train_idx = subject_ids != test_subj
    test_idx  = subject_ids == test_subj
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test   = X[test_idx], y[test_idx]
    
    model = build_model_with_lstm(input_shape=X_train.shape[1:], pretrained_path='datasets_1to5.h5')
    
    early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=8,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Predict
    y_pred = model.predict(X_test, batch_size=16)
    
    # Metrics
    corr_x = np.corrcoef(y_test[:,0], y_pred[:,0])[0,1]
    corr_y = np.corrcoef(y_test[:,1], y_pred[:,1])[0,1]
    r2_x = r2_score(y_test[:,0], y_pred[:,0])
    r2_y = r2_score(y_test[:,1], y_pred[:,1])
    
    print(f"Pearson X: {corr_x:.3f}, Y: {corr_y:.3f} | R2 X: {r2_x:.3f}, Y: {r2_y:.3f}")
    results.append([test_subj, corr_x, corr_y, r2_x, r2_y])
    
    # Plot
    fig, axes = plt.subplots(1,2, figsize=(14,4))
    axes[0].plot(y_test[:,0], label='True X', color='steelblue', alpha=0.8)
    axes[0].plot(y_pred[:,0], label='Pred X', color='lightblue', alpha=0.8)
    axes[0].set_title(f'Subject {test_subj} - X')
    axes[0].legend()
    
    axes[1].plot(y_test[:,1], label='True Y', color='indianred', alpha=0.8)
    axes[1].plot(y_pred[:,1], label='Pred Y', color='salmon', alpha=0.8)
    axes[1].set_title(f'Subject {test_subj} - Y')
    axes[1].legend()
    
    plt.show()

# =========================
# Summary
# =========================
results_df = pd.DataFrame(results, columns=['subject_id','pearson_x','pearson_y','r2_x','r2_y'])
print("\n=== LOSO Summary ===")
print(results_df)


ValueError: Exception encountered when calling layer "avgpool_td" (type TimeDistributed).

Input 0 of layer "average_pooling3d_5" is incompatible with the layer: expected ndim=5, found ndim=7. Full shape received: (None, 1, 47, 29, 18, 1, 16)

Call arguments received by layer "avgpool_td" (type TimeDistributed):
  • inputs=tf.Tensor(shape=(None, 1, 1, 47, 29, 18, 1, 16), dtype=float32)
  • training=None
  • mask=None